# Recirculation box : Pb_Hydraulique

In [ ]:
from trustutils import run
import matplotlib.pyplot as plt
run.introduction("Edouard Butaye & Alan Burlot")
run.TRUST_parameters()

## Problem description

This form allows to compare both $k-\epsilon$ and $k-\omega$ models in a simple channel flow. Both models are tested in both VEF and VDF with implicit and explicit time schemes. Standard $k-\omega$ model should not be used as it depends to much on the freestream condition. Use with caution.

We simulate a single phase turbulent flow in RANS in a periodic channel flow. The channel is $L=0.012\,m$ long and $H=0.66667\,m$ height. The Reynolds number based on the bulk velocity is $7500$.


## Definition of bloc dictionnaries
### Discretisations
The domain uses one block. Both VEF and VDF are tested. For VEF, we use :
- Nx = 7, Ny = 101

For VDF, we use
- Nx = 13, Ny = 201

The grid is refined at the wall. The first mesh is 2mm height, ensuring $y^+<1$ at the wall.

In [ ]:
dic_VEF = {"discretisation": "VEF",
           "triangulate": "trianguler_H dom",
           "nx": 7,
           "ny": 101}
dic_VDF = {"discretisation": "VDF",
           "triangulate": "",
           "nx": 13,
           "ny": 201}

### Time schemes

Both explicit and implicit Euler schemes are tested. For the implicit scheme, a `facsec = 20` is used.

In [ ]:
dic_expl = {
    "scheme": "schema_euler_explicite",
    "scheme_options":
    """
        tinit 0.
        tmax 2
        dt_min 1e-7
        facsec 1
        dt_impr 1
        seuil_statio 1.e-6
    """
}
dic_impl = {
    "scheme": "schema_euler_implicite",
    "scheme_options":
    """
        tinit 0.
        tmax 2
        dt_min 1e-7
        facsec 1
        facsec_max 20
        dt_impr 1
        seuil_statio 1.e-6
        solveur implicite { solveur gmres { diag nb_it_max 3 seuil 1e-12 impr } }
    """
}

WARNING: 10s of computation is not enough for the velocity profil to converge.

### Computation setup

For the $k-\omega$ model, we perform a test for both SST and STD variants.

In [ ]:
komega_variant = ["STD", "BSL", "SST"]

ddis = {
    "VEF": dic_VEF,
    "VDF": dic_VDF
}
dscheme = {
    "EXPL": dic_expl,
    "IMPL": dic_impl
}


### Post-processing

In [ ]:
dfield_post_eps = {
    "field_post":
    """
    format lata
    champs binaire dt_post 1
    {
      vitesse elem
      k elem
      eps elem
      y_plus elem
      pression elem
    }
    """
}
dfield_post_omega = {
    "field_post":
    """
    champs binaire dt_post 1
    {
      vitesse elem
      k elem
      omega elem
      y_plus elem
      pression elem
      production_k elem
      production_omega elem
      dissipation_k elem
      dissipation_omega elem
      cross_diffusion_k_omega elem
      enstrophy elem
      grad_k_grad_omega elem
      tab_F1 elem
      tab_F2 elem
      grad_k elem
      grad_omega elem
      viscosite_turbulente elem
    }
    """
}

In [ ]:
#run.reset()
#run.initBuildDirectory()

# K-epsilon cases
for kdis, vdis in ddis.items():
    for kscheme, vscheme in dscheme.items():
        target_repo = f"{kdis}/{kscheme}/KEPSILON"
        run.addCaseFromTemplate("recirculation_box_k_epsilon.data", target_repo, {**vdis, **vscheme, **dfield_post_eps})

# K-omega cases
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for modvar in komega_variant:
            dico = {"modvar": modvar}
            target_repo = f"{kdis}/{kscheme}/KOMEGA-{modvar}"
            run.addCaseFromTemplate("recirculation_box_k_omega.data", target_repo, {**vdis, **vscheme, **dico, **dfield_post_omega})

run.printCases()


In [ ]:
#run.runCases()

### Performances

In [ ]:
table = run.tablePerf()
table = table.drop(columns=["host", "system"]).drop("Total")
table

## Results
### Residuals

In [ ]:
from trustutils import plot
import itertools

marker = itertools.cycle(('^', '+', 'd', 'o', '*', '<', '>', 's'))

dcolors_k_eps = {"VEF": "blue", "VDF": "red"}
dcolors_k_omega = {"VEF": "cyan", "VDF": "orange"}

dstyle = {"IMPL": "solid", "EXPL": "dashed"}

In [ ]:
nb_cases=len(ddis.keys())*len(dscheme.keys())+len(ddis.keys())*len(dscheme.keys())*len(komega_variant)
cmap = plt.cm.get_cmap('Paired', nb_cases)
colors = cmap(range(nb_cases))

a = plot.Graph("Residuals")
ind_color=0
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        a.addResidu(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KEPSILON/recirculation_box_k_epsilon.dt_ev",
                    label=f"{kdis} {kscheme} KEPSILON ",
                    color=colors[ind_color])
        ind_color+=1

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addResidu(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega.dt_ev",
                        label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                        color=colors[ind_color])
            ind_color+=1

a.scale(yscale='log')

In [ ]:
import numpy as np
y_dns_lamballais,vx_dns_lamballais,vy_dns_lamballais=np.loadtxt("src/velocity_profile_dns_Lamballais_2014.dat",delimiter=" ",unpack=True)

In [ ]:
a = plot.Graph("Vitesse X - Euler explicite")
for kdis in ddis.keys():
    a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/EXPL/KEPSILON/recirculation_box_k_epsilon_VELOCITY_PROBE.son",
                 compo=0,
                 marker = next(marker),
                 label=f"{kdis} KEPSILON",
                 color=dcolors_k_eps[kdis],
                 linestyle=dstyle["EXPL"])
for kdis in ddis.keys():
    for modvar in komega_variant:
        a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/EXPL/KOMEGA-{modvar}/recirculation_box_k_omega_VELOCITY_PROBE.son",
                     compo=0,
                     marker = next(marker),
                     label=f"{kdis} KOMEGA {modvar.upper()}",
                     color=dcolors_k_omega[kdis],
                     linestyle=dstyle["EXPL"])


In [ ]:
a = plot.Graph("Velocity X - Euler implicite")
for kdis in ddis.keys():
        a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/IMPL/KEPSILON/recirculation_box_k_epsilon_VELOCITY_PROBE.son",
                     compo=0,
                     marker = next(marker),
                     label=f"{kdis} {kscheme} KEPSILON",
                     color=dcolors_k_eps[kdis],
                     linestyle=dstyle["IMPL"])

for kdis in ddis.keys():
    for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/IMPL/KOMEGA-{modvar}/recirculation_box_k_omega_VELOCITY_PROBE.son",
                         compo=0,
                         marker = next(marker),
                         label=f"{kdis} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle["IMPL"])
u_out=0.125
plt.plot(y_dns_lamballais,vx_dns_lamballais*u_out, label="DNS Lamballais 2014") #scale to compare (extract from the backward_facing_step) : U_out lamballais = 1m/s. U_out_present_work=0.125


In [ ]:
a = plot.Graph("Velocity Y")

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KEPSILON/recirculation_box_k_epsilon_VELOCITY_PROBE.son",
                         compo=1,
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KEPSILON",
                         color=dcolors_k_eps[kdis],
                         linestyle=dstyle[kscheme])

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_VELOCITY_PROBE.son",
                             compo=1,
                             marker = next(marker),
                             label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                             color=dcolors_k_omega[kdis],
                             linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Viscosity")

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KEPSILON/recirculation_box_k_epsilon_VISCOSITY_PROBE.son",
                     marker = next(marker),
                     label=f"{kdis} {kscheme} KEPSILON",
                     color=dcolors_k_eps[kdis],
                     linestyle=dstyle[kscheme])

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_VISCOSITY_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])



In [ ]:
a = plot.Graph("Viscosity K_Omega SST")

modvar="SST"
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_VISCOSITY_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])


In [ ]:
a = plot.Graph("K")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KEPSILON/recirculation_box_k_epsilon_K_PROBE.son",
                     marker = next(marker),
                     label=f"{kdis} {kscheme} KEPSILON",
                     color=dcolors_k_eps[kdis],
                     linestyle=dstyle[kscheme])

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_K_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])


In [ ]:
a = plot.Graph("K (for K_Omega SST model)")
modvar="SST"
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_K_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])


In [ ]:
a = plot.Graph("Eps")

for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KEPSILON/recirculation_box_k_epsilon_EPS_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KEPSILON",
                         color=dcolors_k_eps[kdis],
                         linestyle=dstyle[kscheme])



In [ ]:
a = plot.Graph("Omega")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_OMEGA_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Omega (for K_Omega SST Model)")
modvar="SST"
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_OMEGA_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Cross Diffusion")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_CKOMEGA_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Production K")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_PK_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Dissipation K")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_DK_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Dissipation K (for K_Omega SST model)")
modvar="SST"
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_DK_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Dissipation Omega")
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
        for modvar in komega_variant:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_DOMEGA_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])

In [ ]:
a = plot.Graph("Dissipation Omega (for K_Omega SST model)")
modvar="SST"
for kdis in ddis.keys():
    for kscheme in dscheme.keys():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/KOMEGA-{modvar}/recirculation_box_k_omega_DOMEGA_PROBE.son",
                         marker = next(marker),
                         label=f"{kdis} {kscheme} KOMEGA {modvar.upper()}",
                         color=dcolors_k_omega[kdis],
                         linestyle=dstyle[kscheme])